# 电商经营与用户价值数据分析 · 探索性分析（EDA）

> 用途：对原始数据做数据质量检查与直观探索，并与 SQL 结论相互印证。
> 语言统一：代码 + 中文解释 + 图表 + 结论。
> 本 notebook 也回答面试常用追问（为何用 customer_unique_id、为何不能直接 SUM 支付表、GMV 为何不直接取支付表）。

数据目录 data/raw/。可先运行 src/data_cleaning.py 生成 data/cleaned/。

In [ ]:
# 1) 载入依赖与原始数据
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = os.getcwd()
RAW = os.path.join(BASE, 'data', 'raw')

def rd(n):
    return pd.read_csv(os.path.join(RAW, f'olist_{n}_dataset.csv'))

customers = rd('customers'); orders = rd('orders'); items = rd('order_items')
payments = rd('order_payments'); reviews = rd('order_reviews')
products = rd('products'); sellers = rd('sellers')
cat = pd.read_csv(os.path.join(RAW, 'product_category_name_translation.csv'))

for c in ['order_purchase_timestamp','order_delivered_customer_date',
          'order_estimated_delivery_date','order_approved_at']:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')
print('数据载入完成')

## 2. 数据质量检查

先看规模、主键唯一性、关键时间字段缺失。这是"结论可不可信"的地基。

In [ ]:
# 2.1 各表规模
for name, d in [('customers',customers),('orders',orders),('order_items',items),
                ('order_payments',payments),('order_reviews',reviews),
                ('products',products),('sellers',sellers),('category_name',cat)]:
    print(f'{name:14s}  rows={len(d):6d}  cols={list(d.columns)[:6]}')

In [ ]:
# 2.2 主键唯一性 & 订单状态分布
print('orders.order_id 去重数 vs 行数:', orders['order_id'].nunique(), len(orders))
print()
print('订单状态分布:')
print(orders['order_status'].value_counts())
print()
print('orders 关键时间缺失:')
print(orders[['order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']].isna().sum())

In [ ]:
# 2.3 重复评价 & 支付 1:多
print('reviews review_id 重复行:', int(reviews['review_id'].duplicated().sum()))
print('reviews 中同 order_id 有多行:', int((reviews.groupby('order_id').size()>1).sum()))
pg = payments.groupby('order_id').size()
print('payments 单笔支付订单数:', int((pg==1).sum()), ' 多笔支付订单数:', int((pg>1).sum()))

结论（与 SQL 的 01_data_quality.sql 印证）：
- 各表主键去重无重复，结构健壮。
- order_status 中 delivered=96,478，canceled/unavailable 等非交付约 2,963 单 → 后续有效口径必须筛 delivered。
- order_payments 约 2,961 单多次支付 → 从支付表直接 SUM 会重复统计 GMV。
- order_reviews 有 814 行重复 review_id、547 单同单多评 → 平均分应按 order_id 聚合再算。

## 3. 经营与用户关键指标（与 SQL 印证）

In [ ]:
# 3.1 有效订单口径下 GMV / AOV
valid = orders[orders['order_status']=='delivered']
vi = items.merge(valid[['order_id']], on='order_id')
gmv = vi['price'].sum(); n_orders = vi['order_id'].nunique()
print('有效订单数:', n_orders)
print('GMV(商品价): %.2f  BRL' % gmv)
print('AOV: %.2f  BRL' % (gmv/n_orders))
print('有效运费合计: %.2f  BRL' % vi['freight_value'].sum())

In [ ]:
# 3.2 复购（真实用户 customer_unique_id）
uo = customers.merge(valid[['customer_id']], on='customer_id') \
        .groupby('customer_unique_id')['customer_id'].nunique()
print('有购买用户:', len(uo))
print('复购用户(>=2单):', int((uo>=2).sum()))
print('复购率: %.2f%%' % ((uo>=2).mean()*100))

关键点：用户分析一律用 customer_unique_id。理由：
- customer_id 是订单维度标识，一单一个；用它会重复数人头。
- 实测 customer_id 去重数≈订单数(99,441)，而 customer_unique_id 去重=96,096，才是真实用户数。
- 用错维度，复购率、Cohort、RFM 全部失真。

## 4. 物流与评分关联（核心洞察的极简验证）

In [ ]:
r = reviews.groupby('order_id')['review_score'].mean().rename('score')
od = valid[['order_id','order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']] \
         .dropna(subset=['order_delivered_customer_date','order_estimated_delivery_date'])
od['days'] = (od['order_delivered_customer_date']-od['order_purchase_timestamp']).dt.days
od = od.merge(r, on='order_id', how='inner')
od['late'] = (od['order_delivered_customer_date']>od['order_estimated_delivery_date']).astype(int)
print('准时均分: %.2f  超时均分: %.2f' % (od.loc[od.late==0,'score'].mean(), od.loc[od.late==1,'score'].mean()))

结论：准时单均分约 4.29，超时单约 2.57，相差约 1.7 分 —— 与 SQL 一致，物流超时与低评分强相关（相关，非因果）。

完整图表与结论见 assets/dashboard/index.html 与 docs/analysis_report.md。